In [1]:
%config InlineBackend.figure_format = 'svg'

In [2]:
import os
import random
import time 
import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import umap.umap_ as umap
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score
from node2vec import Node2Vec
from tqdm import tqdm
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.metrics import CategoricalAccuracy
import torch
from torch_geometric.data import Data
import spektral
from spektral.layers import GCNConv, GATConv
from spektral.layers import GraphSageConv
from spektral.data import Graph, Dataset, BatchLoader
from scipy.sparse import csr_matrix
from spektral.datasets import Cora
from torch_geometric.nn import DeepGraphInfomax, VGAE
from torch_geometric.utils import from_networkx
import scipy.sparse as sp
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from scipy.sparse.csgraph import laplacian
from scipy.sparse.linalg import eigsh
from collections import Counter
from sklearn.preprocessing import normalize
from joblib import Parallel, delayed
from torch_geometric.nn import GCNConv as PyG_GCNConv, VGAE as PyG_VGAE
from torch_geometric.data import Data

In [3]:
SEED = 2025

# Set seed for Python's built-in random module
random.seed(SEED)

# Set seed for NumPy
np.random.seed(SEED)

# Set seed for TensorFlow
tf.random.set_seed(SEED)

# Set seed for PyTorch
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

In [4]:
from torch_geometric.datasets import Planetoid

# Create a custom Dataset for the graph
class CiteSeerDataset(Dataset):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def read(self):
        dataset = Planetoid(root=".", name="CiteSeer")  # Load CiteSeer dataset
        data = dataset[0]  # Access the first graph
        
        # Convert Torch tensors to NumPy
        x = data.x.numpy()
        edge_index = data.edge_index.numpy()
        y = data.y.numpy()

        # One-hot encode labels
        num_classes = y.max() + 1  # Number of classes
        y_one_hot = np.eye(num_classes)[y]  # One-hot encoding

        # Convert edge_index to a sparse adjacency matrix
        num_nodes = x.shape[0]
        adj = csr_matrix((num_nodes, num_nodes))  # Initialize sparse matrix
        for i in range(edge_index.shape[1]):
            src, dst = edge_index[:, i]
            adj[src, dst] = 1
            adj[dst, src] = 1  # Ensure undirected graph

        return [Graph(x=x, a=adj, y=y_one_hot)]

In [5]:
embedding_dimensionality=150

In [6]:
def convert_to_networkx(A):
    return nx.from_scipy_sparse_array(A)

In [7]:
dataset = CiteSeerDataset()
ground_truth_labels = dataset[0].y
labels=np.argmax(ground_truth_labels,axis=1)

C:\Users\user\anaconda3\envs\ContrastiveFUSE\lib\site-packages\torch_geometric\data\dataset.py:238: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  if osp.exists(f) and torch.

In [8]:
mask_file = "C:/Users/user/Benchmarking_fuse/masks/Citeseer/30_70/Citeseer_30_70_masked_indices_seed2025.npy"
labels_to_be_masked = np.load(mask_file)

In [9]:
len(labels_to_be_masked)

2328

In [10]:
masked_labels=[]
for i in np.arange(len(labels)):
    if i in labels_to_be_masked:
        masked_labels.append(-1)
    else:
        masked_labels.append(labels[i])
masked_labels=np.array(masked_labels)

In [11]:
label_mask = masked_labels != -1

In [12]:
X = dataset[0].x
A = dataset[0].a
G = convert_to_networkx(A)

In [13]:
print("Adjacency Matrix Shape:", A.shape)
print("Graph Nodes:", G.number_of_nodes())
print("Graph Edges:", G.number_of_edges())

Adjacency Matrix Shape: (3327, 3327)
Graph Nodes: 3327
Graph Edges: 4552


In [14]:
# Convert your preprocessed data into a PyTorch Geometric Data object
X_py = Data(
    x=torch.tensor(X, dtype=torch.float),  # Node features
    edge_index=torch.tensor(np.array(A.nonzero()), dtype=torch.long),  # Edge indices
    y=torch.tensor(labels, dtype=torch.long)  # Labels
)

# Ensure edge_index is in the correct shape (2, num_edges)
X_py.edge_index = X_py.edge_index.to(torch.long)

## Embeddings

In [15]:
import os
import pickle
import numpy as np
import tensorflow as tf

# -------------------------------------
# Configuration
# -------------------------------------
SEED = 2025
dataset_name = "citeseer"
split_folder = "30-70"

# Input embeddings directory
load_dir = f"C:/Users/user/Benchmarking_fuse/benchmark_outputs/{dataset_name}/{split_folder}/"

# Output save directory
save_dir = f"./citeseer_analysis_results/embeddings/{split_folder.replace('-', '_')}/"
os.makedirs(save_dir, exist_ok=True)

# Embedding filenames to load
embedding_files = {
    "deepwalk": f"deepwalk_embedding_30_70_{SEED}.pkl",
    "node2vec": f"node2vec_embedding_30_70_{SEED}.pkl",
    "fuse": f"fuse_embedding_30_70_{SEED}.pkl",
    "vgae": f"vgae_embedding_30_70_{SEED}.pkl",
    "dgi": f"dgi_embedding_30_70_{SEED}.pkl",
    "random": f"random_embedding_30_70_{SEED}.pkl",
    "given": f"given_embedding_30_70_{SEED}.pkl"
}

# Dictionary to store embeddings
embedding_dict = {}

# -------------------------------------
# Utility: convert numpy to tf.Tensor
# -------------------------------------
def to_tf_tensor(x):
    """Convert numpy array to TensorFlow tensor."""
    if isinstance(x, tf.Tensor):
        return x
    return tf.convert_to_tensor(x, dtype=tf.float32)

# -------------------------------------
# Load and re-save embeddings
# -------------------------------------
for name, filename in embedding_files.items():
    filepath = os.path.join(load_dir, filename)
    if not os.path.exists(filepath):
        print(f" Warning: {name} embedding file not found at {filepath}. Skipping.")
        continue

    print(f" Loading {name.upper()} embedding from {filepath}...")
    with open(filepath, "rb") as f:
        emb = pickle.load(f)

    # Convert to TensorFlow tensor
    embedding_dict[name] = to_tf_tensor(np.array(emb, dtype=float))

    # Save again to new organized directory
    save_path = os.path.join(save_dir, f"{name}_embedding_30_70_{SEED}.pkl")
    with open(save_path, "wb") as f:
        pickle.dump(embedding_dict[name].numpy(), f)
    print(f" Saved {name.upper()} embedding to {save_path}")

print("\n All embeddings loaded into memory and re-saved successfully.")


 Loading DEEPWALK embedding from C:/Users/user/Benchmarking_fuse/benchmark_outputs/citeseer/30-70/deepwalk_embedding_30_70_2025.pkl...
 Saved DEEPWALK embedding to ./citeseer_analysis_results/embeddings/30_70/deepwalk_embedding_30_70_2025.pkl
 Loading NODE2VEC embedding from C:/Users/user/Benchmarking_fuse/benchmark_outputs/citeseer/30-70/node2vec_embedding_30_70_2025.pkl...
 Saved NODE2VEC embedding to ./citeseer_analysis_results/embeddings/30_70/node2vec_embedding_30_70_2025.pkl
 Loading FUSE embedding from C:/Users/user/Benchmarking_fuse/benchmark_outputs/citeseer/30-70/fuse_embedding_30_70_2025.pkl...
 Saved FUSE embedding to ./citeseer_analysis_results/embeddings/30_70/fuse_embedding_30_70_2025.pkl
 Loading VGAE embedding from C:/Users/user/Benchmarking_fuse/benchmark_outputs/citeseer/30-70/vgae_embedding_30_70_2025.pkl...
 Saved VGAE embedding to ./citeseer_analysis_results/embeddings/30_70/vgae_embedding_30_70_2025.pkl
 Loading DGI embedding from C:/Users/user/Benchmarking_fuse/

## Helper functions

In [16]:
def visualize_all_embeddings(all_embeddings, labels, label_mask):
    """
    Visualize all embeddings in a grid with 4 columns per row using UMAP.

    Parameters:
    - all_embeddings: Dictionary where keys are embedding methods, and values are embeddings.
    - labels: Labels (numpy array of shape [n_nodes]).
    - label_mask: Boolean array indicating known labels (True for known, False for unknown).
    """
    num_embeddings = len(all_embeddings)
    num_rows = (num_embeddings + 3) // 4  # Ensure enough rows for all embeddings
    fig, axes = plt.subplots(num_rows, 4, figsize=(8.27, 11.69))  # A4 size

    for i, (embedding_type, embedding) in tqdm(enumerate(all_embeddings.items()), 
                                               total=num_embeddings, desc="Visualizing embeddings"):
        row, col = divmod(i, 4)
        ax = axes[row, col] if num_rows > 1 else axes[col]  # Adjust for single-row case

        # Ensure embedding is a NumPy array
        if isinstance(embedding, tf.Tensor):
            embedding = embedding.numpy()

        # Reduce dimensionality using UMAP
        reducer = umap.UMAP(n_components=2)
        embedding_2d = reducer.fit_transform(embedding)

        # Known labels
        ax.scatter(embedding_2d[label_mask, 0], embedding_2d[label_mask, 1], 
                   c=labels[label_mask], cmap="Set1", s=3, alpha=0.7, label="Known Labels",
                   edgecolors='none')

        # Unknown labels
        ax.scatter(embedding_2d[~label_mask, 0], embedding_2d[~label_mask, 1], 
                   c=labels[~label_mask], cmap="Set1", s=5, alpha=0.7, 
                   label="Unknown Labels", edgecolors='black', linewidths=0.2)

        # Title with smaller font size
        ax.set_title(embedding_type.upper(), fontsize=8, pad=2)

        # Remove axis labels, ticks, and frames
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_frame_on(False)

    # Remove empty subplots if num_embeddings is not a multiple of 4
    for j in range(i + 1, num_rows * 4):
        row, col = divmod(j, 4)
        fig.delaxes(axes[row, col])

    plt.subplots_adjust(left=0.05, right=0.95, top=0.95, bottom=0.05, wspace=0.2, hspace=0.2)  # Adjust margins
    save_path = "./citeseer_analysis_results/embedding_grid_plot_citeseer_30_70.png"
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"Visualization saved to {save_path}")
    plt.show()

In [17]:
def evaluate_model(true_labels, predicted_labels):
    """
    Evaluate the model's performance using accuracy, F1-score, and confusion matrix.

    Args:
        true_labels (np.array): Ground truth labels (integers).
        predicted_labels (np.array): Predicted labels (integers).

    Returns:
        dict: A dictionary containing accuracy, F1-score, and confusion matrix.
    """
    # Compute accuracy
    accuracy = accuracy_score(true_labels, predicted_labels)
    
    # Compute F1-score (macro-averaged)
    f1 = f1_score(true_labels, predicted_labels, average='macro')
    
    # Compute confusion matrix
    cm = confusion_matrix(true_labels, predicted_labels)

    #
    print(cm)
    
    # Return results as a dictionary
    return {
        'accuracy': accuracy,
        'f1_score': f1
    }

## Classifiers

In [18]:
class NoMaskGCNConv(GCNConv):
    def compute_mask(self, inputs, mask=None):
        return None

    def call(self, inputs, training=None, mask=None):
        # Explicitly discard mask
        return super().call(inputs, mask=None)
        
class GCN(tf.keras.Model):
    def __init__(self, n_labels, seed=40):
        super().__init__()
        initializer = tf.keras.initializers.GlorotUniform(seed=seed)
        self.conv1 = NoMaskGCNConv(16, activation='relu', kernel_initializer=initializer)
        self.conv2 = NoMaskGCNConv(n_labels, activation='softmax', kernel_initializer=initializer)

    def call(self, inputs, training=False):
        x, a = inputs
        intermediate_embeddings = self.conv1([x, a])
        x = self.conv2([intermediate_embeddings, a])
        return x, intermediate_embeddings


In [19]:
from spektral.layers import GATConv
import tensorflow as tf

# Define a custom wrapper for GATConv that avoids mask issues
class NoMaskGATConv(GATConv):
    def compute_mask(self, inputs, mask=None):
        return None

    def call(self, inputs, training=None, mask=None):
        # Explicitly discard the mask argument
        return super().call(inputs, mask=None)


# Define the GAT model using the NoMaskGATConv
class GAT(tf.keras.Model):
    def __init__(self, n_labels, num_heads=8, seed=40):
        super().__init__()
        initializer = tf.keras.initializers.GlorotUniform(seed=seed)

        # Use the custom NoMaskGATConv instead of the original GATConv
        self.conv1 = NoMaskGATConv(16, attn_heads=num_heads, concat_heads=True, activation='elu', kernel_initializer=initializer)
        self.conv2 = NoMaskGATConv(n_labels, attn_heads=1, concat_heads=False, activation='softmax', kernel_initializer=initializer)

    def call(self, inputs):
        x, a = inputs
        intermediate_embeddings = self.conv1([x, a])  # Store intermediate embeddings
        x = self.conv2([intermediate_embeddings, a])
        return x, intermediate_embeddings  # Return both final output and intermediate embeddings


In [20]:
# Define the GraphSAGE model
class GraphSAGE(tf.keras.Model):
    def __init__(self, n_labels, hidden_dim=16, aggregator='mean', seed=40):
        super().__init__()
        initializer = tf.keras.initializers.GlorotUniform(seed=seed)

        self.conv1 = GraphSageConv(hidden_dim, activation='relu', aggregator=aggregator, kernel_initializer=initializer)
        self.conv2 = GraphSageConv(n_labels, activation='softmax', aggregator=aggregator, kernel_initializer=initializer)

    def call(self, inputs):
        x, a = inputs
        intermediate_embeddings = self.conv1([x, a])  # Store intermediate embeddings
        x = self.conv2([intermediate_embeddings, a])
        return x, intermediate_embeddings  # Return both final output and intermediate embeddings

In [21]:
classifiers=['gcn','gat','graphsage']

## Classification using different node embeddings

In [23]:
def train_and_evaluate(
    embedding_dict, embedding, classifier,
    ground_truth_labels=ground_truth_labels,
    masked_labels=masked_labels
):
    """
    Version that trains ONLY on the training subgraph:
    - Uses X_train and A_train
    - No test nodes seen during training
    - Inference happens on full graph
    """

    print(f"\nEmbedding: {embedding.upper()}")
    print(f"Model: {classifier.upper()}")

    # -------------------------------------------------
    # 1. Construct training subgraph
    # -------------------------------------------------
    train_mask = masked_labels != -1
    train_idx = np.where(train_mask)[0]

    # Features for training nodes only
    X_train = tf.convert_to_tensor(embedding_dict[embedding][train_mask], dtype=tf.float32)

    # Labels for training (one-hot)
    Y_train = tf.convert_to_tensor(ground_truth_labels[train_mask], dtype=tf.float32)

    # Build training adjacency
    A_train = A[train_mask][:, train_mask]   # induced subgraph

    A_coo = A_train.tocoo()
    A_train_tensor = tf.sparse.SparseTensor(
        indices = np.column_stack([A_coo.row, A_coo.col]),
        values  = A_coo.data.astype(np.float32),
        dense_shape = A_coo.shape
    )
    A_train_tensor = tf.sparse.reorder(A_train_tensor)

    # -------------------------------------------------
    # 2. Build classifier on TRAIN SUBGRAPH only
    # -------------------------------------------------
    n_classes = ground_truth_labels.shape[1]

    if classifier == 'gcn':
        model = GCN(n_classes)
    elif classifier == 'gat':
        model = GAT(n_classes)
    elif classifier == 'graphsage':
        model = GraphSAGE(n_classes)
    else:
        raise ValueError("Unknown classifier: " + classifier)

    optimizer = Adam(learning_rate=0.01)
    loss_fn = CategoricalCrossentropy()

    # -------------------------------------------------
    # 3. Training (on training subgraph only)
    # -------------------------------------------------
    print("Training on TRAINING SUBGRAPH ONLY...")

    epochs = 200
    for epoch in range(epochs):
        with tf.GradientTape() as tape:

            preds_train, _ = model([X_train, A_train_tensor], training=True)

            loss = loss_fn(Y_train, preds_train)

        grads = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))

        if epoch % 20 == 0:
            acc = CategoricalAccuracy()(Y_train, preds_train).numpy()
            print(f"Epoch {epoch} | Loss={loss.numpy():.4f} | Train Acc={acc:.4f}")

    # -------------------------------------------------
    # 4. Inference — NOW use the full graph
    # -------------------------------------------------
    print("Predicting on FULL GRAPH...")

    X_full = tf.convert_to_tensor(embedding_dict[embedding], dtype=tf.float32)

    A_full = A
    A_coo = A_full.tocoo()
    A_full_tensor = tf.sparse.SparseTensor(
        indices = np.column_stack([A_coo.row, A_coo.col]),
        values  = A_coo.data.astype(np.float32),
        dense_shape = A_coo.shape
    )
    A_full_tensor = tf.sparse.reorder(A_full_tensor)

    preds_all, emb_full = model([X_full, A_full_tensor], training=False)

    predicted_labels = tf.argmax(preds_all, axis=1).numpy()

    # Evaluate only on masked nodes (test nodes)
    predicted_test = predicted_labels[labels_to_be_masked]
    true_test = labels[labels_to_be_masked]

    results = evaluate_model(true_test, predicted_test)

    print(f"Accuracy: {results['accuracy']*100:.2f}%")
    print(f"F1 Score: {results['f1_score']:.4f}")

    results["embedding"] = embedding
    results["model"] = classifier

    return results, emb_full


In [24]:
all_results=[]
graph_embeddings_dict={}
for emb in embedding_dict.keys():
    for clf in classifiers:
        results, embedding_matrix = train_and_evaluate(embedding_dict, emb, clf)
        all_results.append(results)
        key_string= emb + ' with ' + clf
        graph_embeddings_dict[key_string]=embedding_matrix


Embedding: DEEPWALK
Model: GCN
Training on TRAINING SUBGRAPH ONLY...
Epoch 0 | Loss=2.0992 | Train Acc=0.1502
Epoch 20 | Loss=1.3801 | Train Acc=0.4244
Epoch 40 | Loss=1.2385 | Train Acc=0.4885
Epoch 60 | Loss=1.1377 | Train Acc=0.5165
Epoch 80 | Loss=1.0763 | Train Acc=0.5425
Epoch 100 | Loss=1.0372 | Train Acc=0.5616
Epoch 120 | Loss=1.0124 | Train Acc=0.5656
Epoch 140 | Loss=0.9952 | Train Acc=0.5716
Epoch 160 | Loss=0.9818 | Train Acc=0.5806
Epoch 180 | Loss=0.9720 | Train Acc=0.5816
Predicting on FULL GRAPH...
[[ 33  68  20  24  26  19]
 [ 37 222  63  30  35  30]
 [ 14  96 282  42  14  27]
 [ 43 118  72 209  27  26]
 [ 23  51  20  28 256  33]
 [  8  70  33  27  26 176]]
Accuracy: 50.60%
F1 Score: 0.4790

Embedding: DEEPWALK
Model: GAT
Training on TRAINING SUBGRAPH ONLY...
Epoch 0 | Loss=1.7874 | Train Acc=0.1652
Epoch 20 | Loss=1.3029 | Train Acc=0.4995
Epoch 40 | Loss=1.1205 | Train Acc=0.5656
Epoch 60 | Loss=1.0549 | Train Acc=0.5706
Epoch 80 | Loss=0.9863 | Train Acc=0.5926
Ep

## Saving aggregate results

In [25]:
# Convert to DataFrame
df = pd.DataFrame(all_results)

# Define dataset name and seed
dataset_name = "citeseer"
seed_value = SEED

# Save as CSV file without sorting
filename = f"{dataset_name}_30_70_{SEED}_results.csv"
filename='./citeseer_analysis_results/results/30_70/'+filename
df.to_csv(filename, index=False)

print(f"Results saved as {filename}")

Results saved as ./citeseer_analysis_results/results/30_70/citeseer_30_70_2025_results.csv


In [26]:
all_embeddings= embedding_dict | graph_embeddings_dict

In [27]:
def reorder_dict(original_dict, key_order):
    """
    Reorders a dictionary based on a given list of keys.

    Parameters:
    - original_dict (dict): The dictionary to reorder.
    - key_order (list): The list specifying the desired key order.

    Returns:
    - dict: A new dictionary with keys ordered as per key_order.
    """
    return {key: original_dict[key] for key in key_order if key in original_dict}

In [28]:
key_order = ['random', 'random with gcn', 'random with gat', 'random with graphsage', 'deepwalk', 'deepwalk with gcn', 'deepwalk with gat', 'deepwalk with graphsage', 'node2vec','node2vec with gcn', 'node2vec with gat', 'node2vec with graphsage', 'vgae', 'vgae with gcn', 'vgae with gat', 'vgae with graphsage', 'dgi', 'dgi with gcn', 'dgi with gat', 'dgi with graphsage', 'fuse', 'fuse with gcn', 'fuse with gat', 'fuse with graphsage', 'given', 'given with gcn', 'given with gat', 'given with graphsage']

In [29]:
all_embeddings = reorder_dict(all_embeddings, key_order)

In [30]:
# visualize_all_embeddings(all_embeddings, labels, label_mask)

In [31]:
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    normalized_mutual_info_score,
    adjusted_rand_score,
    v_measure_score
)
import pandas as pd
# from benchmarking_utils import load_dataset  # or your label loader

# Load true labels
# ds = load_dataset("cora")
# labels = ds["labels"]
ds=dataset
# Choose which embedding sources to evaluate
sources = {
    "Raw Embeddings": embedding_dict,
    "GNN Embeddings": graph_embeddings_dict  # << the key fix
}

metrics_results = []

for source_name, emb_source in sources.items():
    print(f"\n Evaluating {source_name}...\n")

    for name, emb_tensor in emb_source.items():
        emb = emb_tensor.numpy() if isinstance(emb_tensor, tf.Tensor) else np.array(emb_tensor)

        n_clusters = len(np.unique(labels))
        kmeans = KMeans(n_clusters=n_clusters, random_state=SEED, n_init=10)
        cluster_labels = kmeans.fit_predict(emb)

        try:
            silhouette = silhouette_score(emb, cluster_labels)
        except Exception:
            silhouette = np.nan
        try:
            db_index = davies_bouldin_score(emb, cluster_labels)
        except Exception:
            db_index = np.nan

        nmi = normalized_mutual_info_score(labels, cluster_labels)
        ari = adjusted_rand_score(labels, cluster_labels)
        v_measure = v_measure_score(labels, cluster_labels)

        metrics_results.append({
            "Source": source_name,
            "Embedding": name,
            "Silhouette": silhouette,
            "Davies-Bouldin": db_index,
            "NMI": nmi,
            "ARI": ari,
            "V-Measure": v_measure
        })

        print(f" {name.upper()} | Silhouette={silhouette:.4f}, DB={db_index:.4f}, NMI={nmi:.4f}, ARI={ari:.4f}, V={v_measure:.4f}")

# Save results
metrics_df = pd.DataFrame(metrics_results)
metrics_path = os.path.join(save_dir, f"clustering_metrics_combined_{dataset_name}_{split_folder.replace('-', '_')}_{SEED}.csv")
metrics_df.to_csv(metrics_path, index=False)
print(f"\n Combined clustering metrics saved to: {metrics_path}")
print(metrics_df)



🔍 Evaluating Raw Embeddings...

 DEEPWALK | Silhouette=0.0578, DB=3.8248, NMI=0.2004, ARI=0.1017, V=0.2004
 NODE2VEC | Silhouette=0.0501, DB=4.2505, NMI=0.1999, ARI=0.1083, V=0.1999
 FUSE | Silhouette=0.3379, DB=7.9979, NMI=0.0623, ARI=0.0297, V=0.0623
 VGAE | Silhouette=0.0379, DB=4.6632, NMI=0.1304, ARI=0.0533, V=0.1304
 DGI | Silhouette=0.3655, DB=0.8477, NMI=0.0232, ARI=0.0122, V=0.0232
 RANDOM | Silhouette=0.0054, DB=9.1165, NMI=0.0017, ARI=-0.0004, V=0.0017
 GIVEN | Silhouette=0.0053, DB=8.4735, NMI=0.2750, ARI=0.2271, V=0.2750

🔍 Evaluating GNN Embeddings...

 DEEPWALK WITH GCN | Silhouette=0.4845, DB=1.0343, NMI=0.0682, ARI=0.0032, V=0.0682
 DEEPWALK WITH GAT | Silhouette=0.1558, DB=2.0543, NMI=0.3096, ARI=0.2316, V=0.3096
 DEEPWALK WITH GRAPHSAGE | Silhouette=0.2403, DB=1.4843, NMI=0.2706, ARI=0.1415, V=0.2706
 NODE2VEC WITH GCN | Silhouette=0.3917, DB=1.0486, NMI=0.0705, ARI=0.0159, V=0.0705
 NODE2VEC WITH GAT | Silhouette=0.1196, DB=2.3845, NMI=0.2752, ARI=0.2066, V=0.2752


C:\Users\user\anaconda3\envs\ContrastiveFUSE\lib\site-packages\sklearn\base.py:1473: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (6). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


 DGI WITH GAT | Silhouette=0.2900, DB=1.0324, NMI=0.0361, ARI=0.0199, V=0.0361
 DGI WITH GRAPHSAGE | Silhouette=0.2471, DB=1.1839, NMI=0.0203, ARI=0.0076, V=0.0203
 RANDOM WITH GCN | Silhouette=0.1080, DB=2.3962, NMI=0.0117, ARI=0.0012, V=0.0117
 RANDOM WITH GAT | Silhouette=0.0125, DB=5.2370, NMI=0.0332, ARI=0.0216, V=0.0332
 RANDOM WITH GRAPHSAGE | Silhouette=0.1032, DB=2.1691, NMI=0.0036, ARI=-0.0001, V=0.0036
 GIVEN WITH GCN | Silhouette=0.4348, DB=1.0642, NMI=0.0318, ARI=-0.0005, V=0.0318
 GIVEN WITH GAT | Silhouette=0.2794, DB=1.3230, NMI=0.5240, ARI=0.5416, V=0.5240
 GIVEN WITH GRAPHSAGE | Silhouette=0.3272, DB=1.2578, NMI=0.5194, ARI=0.5467, V=0.5194

📊 Combined clustering metrics saved to: ./citeseer_analysis_results/embeddings/30_70/clustering_metrics_combined_citeseer_30_70_2025.csv
            Source                Embedding  Silhouette  Davies-Bouldin  \
0   Raw Embeddings                 deepwalk    0.057816        3.824772   
1   Raw Embeddings                 node2vec  

In [37]:
import pandas as pd
import os
import numpy as np

# ----------------------------------------
# Configuration
# ----------------------------------------
dataset_name = "citeseer"
split_folder = "30_70"
seeds = [42, 46, 123, 999, 2025]
base_dir = f"./citeseer_analysis_results/embeddings/{split_folder}/"

# Output file
output_path = os.path.join(base_dir, f"clustering_metrics_combined_{dataset_name}_{split_folder}_averaged.csv")

# ----------------------------------------
# Load and combine all CSVs
# ----------------------------------------
dfs = []
for seed in seeds:
    filename = f"clustering_metrics_combined_{dataset_name}_{split_folder}_{seed}.csv"
    filepath = os.path.join(base_dir, filename)
    
    if os.path.exists(filepath):
        df = pd.read_csv(filepath)
        df["Seed"] = seed
        dfs.append(df)
        print(f" Loaded: {filepath}")
    else:
        print(f" Missing file for seed {seed}: {filepath}")

if not dfs:
    raise FileNotFoundError("No CSV files found — check your directory and filenames.")

# Merge all seed data
combined_df = pd.concat(dfs, ignore_index=True)

# ----------------------------------------
# Average metrics across seeds
# ----------------------------------------
agg_df = (
    combined_df
    .groupby(["Source", "Embedding"])
    .agg({
        "Silhouette": ["mean", "std"],
        "Davies-Bouldin": ["mean", "std"],
        "NMI": ["mean", "std"],
        "ARI": ["mean", "std"],
        "V-Measure": ["mean", "std"]
    })
    .reset_index()
)

# Flatten column names
agg_df.columns = [
    "Source", "Embedding",
    "Silhouette_Mean", "Silhouette_STD",
    "DB_Mean", "DB_STD",
    "NMI_Mean", "NMI_STD",
    "ARI_Mean", "ARI_STD",
    "VMeasure_Mean", "VMeasure_STD"
]

# Save the averaged results
agg_df.to_csv(output_path, index=False)
print(f"\n Averaged metrics saved to: {output_path}\n")

# Display summary
print(agg_df)


✅ Loaded: ./citeseer_analysis_results/embeddings/30_70/clustering_metrics_combined_citeseer_30_70_42.csv
✅ Loaded: ./citeseer_analysis_results/embeddings/30_70/clustering_metrics_combined_citeseer_30_70_46.csv
✅ Loaded: ./citeseer_analysis_results/embeddings/30_70/clustering_metrics_combined_citeseer_30_70_123.csv
✅ Loaded: ./citeseer_analysis_results/embeddings/30_70/clustering_metrics_combined_citeseer_30_70_999.csv
✅ Loaded: ./citeseer_analysis_results/embeddings/30_70/clustering_metrics_combined_citeseer_30_70_2025.csv

📊 Averaged metrics saved to: ./citeseer_analysis_results/embeddings/30_70/clustering_metrics_combined_citeseer_30_70_averaged.csv

            Source                Embedding  Silhouette_Mean  Silhouette_STD  \
0   GNN Embeddings        deepwalk with gat         0.138693        0.011631   
1   GNN Embeddings        deepwalk with gcn         0.482555        0.008660   
2   GNN Embeddings  deepwalk with graphsage         0.236790        0.021600   
3   GNN Embeddings 